# QM9 loader statistics

This notebook inspects one selected `QM9Loader` CSV export and draws graph-level and target-level statistics for that dataset. In practice the common workflow is to work with the standard `qm9` export rather than to browse a multi-dataset catalog.


## Data preparation

This workflow downloads `qm9.csv` when it is missing, stores it under `data-local/QM9/`, and builds graph caches under `data-local/graph_corpus_cache/`. The parsed graph cache is reused on reruns.


In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from abstractgraph_graphicalizer.chem import (
    DEFAULT_QM9_TARGET_COLUMNS,
    QM9Loader,
    download_qm9_dataset,
)


## Resolve one dataset

This notebook starts from one explicit dataset name. If that CSV is not present locally, it can download the default `qm9.csv` export automatically.


In [ ]:
qm9_root = repo_root / "data-local" / "QM9"
loader = QM9Loader(qm9_root, on_error="skip")
dataset_name = "qm9"
auto_download_if_missing = True
download_filename = f"{dataset_name}.csv"

try:
    paths = loader.resolve_paths(dataset_name)
except FileNotFoundError:
    loader.root.mkdir(parents=True, exist_ok=True)
    if auto_download_if_missing:
        csv_path = download_qm9_dataset(loader.root, filename=download_filename)
        print(f"Downloaded {csv_path.name} to {csv_path.parent}")
        paths = loader.resolve_paths(dataset_name)
    else:
        raise FileNotFoundError(
            f"Dataset {dataset_name!r} was not found under {loader.root}. "
            "Set auto_download_if_missing=True or place the CSV under that root."
        )

summary = next(
    summary
    for summary in loader.list_datasets()
    if summary.dataset_name == dataset_name
)
dataset_overview = pd.DataFrame(
    [
        {
            "dataset_name": summary.dataset_name,
            "molecule_count": summary.molecule_count,
            "size_mb": summary.size_mb,
            "csv_path": str(summary.csv_path),
        }
    ]
)

print("root:", loader.root)
display(dataset_overview)


## Load the selected dataset and inspect graph statistics

The selected CSV is loaded through the cached graph-corpus path owned by `QM9Loader.load(...)`.


In [ ]:
limit = None
max_node_count = None

graphs, metadata = loader.load(
    dataset_name,
    limit=limit,
    max_node_count=max_node_count,
)

print("dataset_name:", dataset_name)
print("csv_path:", paths.csv_path)
print("graphs loaded:", len(graphs))
display(metadata.head())


In [ ]:
graph_stats = pd.DataFrame(
    {
        "node_count": [graph.number_of_nodes() for graph in graphs],
        "edge_count": [graph.number_of_edges() for graph in graphs],
    }
)
graph_stats["average_degree"] = 2.0 * graph_stats["edge_count"] / graph_stats["node_count"].clip(lower=1)
graph_stats["density"] = 2.0 * graph_stats["edge_count"] / (
    graph_stats["node_count"] * (graph_stats["node_count"] - 1)
).where(graph_stats["node_count"] > 1, 1)

candidate_target_columns = [column for column in DEFAULT_QM9_TARGET_COLUMNS if column in metadata.columns]
display(graph_stats.head())
graph_stats.describe().transpose()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 10))

node_tick_values = np.arange(graph_stats["node_count"].min(), graph_stats["node_count"].max() + 1, 1)
node_bins = np.arange(graph_stats["node_count"].min() - 0.5, graph_stats["node_count"].max() + 1.5, 1)
edge_tick_values = np.arange(graph_stats["edge_count"].min(), graph_stats["edge_count"].max() + 1, 1)
edge_bins = np.arange(graph_stats["edge_count"].min() - 0.5, graph_stats["edge_count"].max() + 1.5, 1)

axes[0, 0].hist(graph_stats["node_count"], bins=node_bins, color="#1f77b4", alpha=0.8)
axes[0, 0].set_yscale("log")
axes[0, 0].set_xticks(node_tick_values)
axes[0, 0].set_title("Node count distribution")
axes[0, 0].set_xlabel("nodes")
axes[0, 0].set_ylabel("graphs")

axes[0, 1].hist(graph_stats["edge_count"], bins=edge_bins, color="#ff7f0e", alpha=0.8)
axes[0, 1].set_yscale("log")
axes[0, 1].set_xticks(edge_tick_values)
axes[0, 1].set_title("Edge count distribution")
axes[0, 1].set_xlabel("edges")
axes[0, 1].set_ylabel("graphs")

axes[1, 0].hist(graph_stats["average_degree"], bins=10, log=True, color="#2ca02c", alpha=0.8)
axes[1, 0].set_title("Average degree distribution")
axes[1, 0].set_xlabel("average degree")
axes[1, 0].set_ylabel("graphs")

axes[1, 1].scatter(graph_stats["node_count"], graph_stats["edge_count"], s=20, alpha=0.45, color="#d62728")
axes[1, 1].set_title("Edges vs nodes")
axes[1, 1].set_xlabel("nodes")
axes[1, 1].set_ylabel("edges")

plt.tight_layout()


In [ ]:
summary = pd.DataFrame(
    {
        "graphs": [len(graph_stats)],
        "mean_nodes": [graph_stats["node_count"].mean()],
        "median_nodes": [graph_stats["node_count"].median()],
        "mean_edges": [graph_stats["edge_count"].mean()],
        "median_edges": [graph_stats["edge_count"].median()],
        "mean_density": [graph_stats["density"].mean()],
    },
    index=[dataset_name],
).round(3)

summary


## QM9 target statistics

QM9 includes a standard family of molecular property targets. The next cells summarize the available target columns and draw histograms for a compact subset.


In [ ]:
target_frame = metadata.loc[:, candidate_target_columns].copy()
display(target_frame.describe().transpose())

selected_target_columns = [column for column in ["mu", "alpha", "gap", "cv", "u0"] if column in target_frame.columns]
selected_target_columns


In [ ]:
if selected_target_columns:
    fig, axes = plt.subplots(1, len(selected_target_columns), figsize=(5 * len(selected_target_columns), 4))
    if len(selected_target_columns) == 1:
        axes = [axes]

    for axis, column in zip(axes, selected_target_columns):
        axis.hist(target_frame[column].dropna(), bins=30, color="#9467bd", alpha=0.8)
        axis.set_title(f"{column} distribution")
        axis.set_xlabel(column)
        axis.set_ylabel("molecules")

    plt.tight_layout()
else:
    print("No QM9 target columns found in this dataset:", DEFAULT_QM9_TARGET_COLUMNS)
